# TP02 — OOP
## Exercise 1: Build the `CSVReader` Class

### Objective
This exercise practices the basic concepts of Object-Oriented Programming (OOP):

- **Class** A class is a template for creating objects.
- **Constructor (`__init__`)** A constructor is the special method that runs automatically when an object is created.
- **Methods** A method is a function that belongs to a class.
- **Encapsulation** Encapsulation means keeping related data and functions together inside one class.

The `CSVReader` class will keep all CSV-reading operations inside one reusable class.

## 1 Import pandas

We use **pandas** because `pandas.read_csv()` can load a CSV file into a DataFrame.

In [1]:
import pandas as pd

## 2 — Create the `CSVReader` Class

The class contains:

- `__init__(file_path)` → stores the CSV file path. A constructor is a special method that runs automatically when you create a new object.
- `read()` → reads the CSV file using pandas.
- `preview(n)` → prints the first `n` rows.

`self.data` stores the loaded DataFrame so that other methods can reuse it.

In [2]:
class CSVReader:
    """A simple reusable class for reading and previewing CSV files."""

    def __init__(self, file_path):
        """Store the path of the CSV file."""
        self.file_path = file_path
        self.data = None

    def read(self):
        """Read the CSV file and store it in self.data."""
        self.data = pd.read_csv(self.file_path)
        return self.data

    def preview(self, n=5):
        """Print the first n rows of the loaded CSV file."""
        if self.data is None:
            self.read()

        print(self.data.head(n))

## 3 — Create an Object

The CSV file is in the same folder as this notebook, so we pass its filename to `CSVReader`.

In [3]:
reader = CSVReader("sample_data (2).csv")

## 4 — Read the CSV File

Calling `read()` loads the CSV file and returns a pandas DataFrame.

In [4]:
df = reader.read()
df

,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,29.0,165.0,68.0,New York,85.0
1,2,Bob,NaN,172.0,NaN,Los Angeles,90.0
2,3,Charlie,35.0,168.0,72.0,Chicago,NaN
3,4,David,NaN,NaN,80.0,Houston,75.0
4,5,Eva,27.0,160.0,55.0,New York,88.0
5,6,Frank,30.0,175.0,85.0,Los Angeles,NaN
6,7,Grace,NaN,162.0,60.0,Chicago,92.0


## 5 — Preview the First 5 Rows

The `preview()` method displays the top rows of the dataset.  
By default, `n=5`.

In [5]:
reader.preview(5)

   id     name   age  height_cm  weight_kg         city  score
0   1    Alice  29.0      165.0       68.0     New York   85.0
1   2      Bob   NaN      172.0        NaN  Los Angeles   90.0
2   3  Charlie  35.0      168.0       72.0      Chicago    NaN
3   4    David   NaN        NaN       80.0      Houston   75.0
4   5      Eva  27.0      160.0       55.0     New York   88.0


## 6 — Preview a Different Number of Rows

Because `preview(n)` accepts an argument, we can choose how many rows to display.

In [6]:
reader.preview(3)

   id     name   age  height_cm  weight_kg         city  score
0   1    Alice  29.0      165.0       68.0     New York   85.0
1   2      Bob   NaN      172.0        NaN  Los Angeles   90.0
2   3  Charlie  35.0      168.0       72.0      Chicago    NaN


## Result

The `CSVReader` class successfully:

1. Stores the CSV file path.
2. Reads the file with pandas.
3. Stores the DataFrame inside the object.
4. Previews any requested number of rows.

This completes **Exercise 1**.

# Exercise 2: Apply Pattern for Missing Values Handling

## Objective

- Learn the Strategy Pattern: one interface, many interchangeable behaviors.
- Practice inheritance and polymorphism.

## Concept

Different strategies can be used to handle missing data, such as dropping missing values, filling them with the mean, or filling them with the mode.

The Strategy Pattern allows us to switch between these methods without changing the main code.

## Instructions:
1. Create an abstract class MissingValueStrategy with a method handle(df).
2. Create subclasses:
- oDropMissing
- oFillMean
- oFillMode
3. Create a DataCleaner class that applies the chosen strategy.

In [7]:
import pandas as pd

df = pd.read_csv("D:\Advanced-Programming-TP\sample_data (2).csv")
print(df.head())

   id     name   age  height_cm  weight_kg         city  score
0   1    Alice  29.0      165.0       68.0     New York   85.0
1   2      Bob   NaN      172.0        NaN  Los Angeles   90.0
2   3  Charlie  35.0      168.0       72.0      Chicago    NaN
3   4    David   NaN        NaN       80.0      Houston   75.0
4   5      Eva  27.0      160.0       55.0     New York   88.0


In [1]:
# 1. Create an abstract class MissingValueStrategy with a method handle(df).
from abc import ABC, abstractmethod


class MissingValueStrategy(ABC):

    @abstractmethod
    def handle(self, df):
        pass

In [2]:
# 2. Create subclasses:
# DropMissing
class DropMissing(MissingValueStrategy):
    def handle(self, df):
        return df.dropna()

In [8]:
df = pd.read_csv("D:\Advanced-Programming-TP\sample_data (2).csv")

cleaner = DataCleaner(DropMissing())
cleaned_df = cleaner.clean(df)

print(cleaned_df.head())

   id   name   age  height_cm  weight_kg      city  score
0   1  Alice  29.0      165.0       68.0  New York   85.0
4   5    Eva  27.0      160.0       55.0  New York   88.0


In [5]:
# FillMean
class FillMean(MissingValueStrategy):
    def handle(self, df):
        result = df.copy()
        numeric_cols = result.select_dtypes(include="number").columns
        result[numeric_cols] = result[numeric_cols].fillna(
            result[numeric_cols].mean()
        )
        return result

In [9]:
df = pd.read_csv("D:\Advanced-Programming-TP\sample_data (2).csv")

cleaner = DataCleaner(FillMean())
cleaned_df = cleaner.clean(df)

print(cleaned_df.head())

   id     name    age  height_cm  weight_kg         city  score
0   1    Alice  29.00      165.0       68.0     New York   85.0
1   2      Bob  30.25      172.0       70.0  Los Angeles   90.0
2   3  Charlie  35.00      168.0       72.0      Chicago   86.0
3   4    David  30.25      167.0       80.0      Houston   75.0
4   5      Eva  27.00      160.0       55.0     New York   88.0


In [4]:
# FillMode
class FillMode(MissingValueStrategy):
    def handle(self, df):
        result = df.copy()

        for column in result.columns:
            if result[column].isna().any():
                mode_values = result[column].mode()

                if not mode_values.empty:
                    result[column] = result[column].fillna(
                        mode_values.iloc[0]
                    )

        return result

In [10]:
df = pd.read_csv("D:\Advanced-Programming-TP\sample_data (2).csv")

cleaner = DataCleaner(FillMode())
cleaned_df = cleaner.clean(df)

print(cleaned_df.head())

   id     name   age  height_cm  weight_kg         city  score
0   1    Alice  29.0      165.0       68.0     New York   85.0
1   2      Bob  27.0      172.0       55.0  Los Angeles   90.0
2   3  Charlie  35.0      168.0       72.0      Chicago   75.0
3   4    David  27.0      160.0       80.0      Houston   75.0
4   5      Eva  27.0      160.0       55.0     New York   88.0


In [11]:
# 3. Create a DataCleaner class that applies the chosen strategy.
class DataCleaner:
    def __init__(self, strategy):
        self.strategy = strategy

    def clean(self, df):
        return self.strategy.handle(df)

# Exercise 3: Add Decorators for Logging and Timing

**Objectives**
- Learn the **Decorator Pattern** to add reusable behaviour (logging, timing).
- Understand how decorators support the **Open/Closed Principle**: classes stay *closed for modification* but *open for extension*.

**Concept:** a decorator is a function that takes a function, wraps it with extra behaviour, and returns the wrapper. The original method body is never edited.

In [1]:
import sys
import time
import logging
import functools
import pandas as pd

# Configure logging once so the decorators print to the notebook
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
    stream=sys.stdout,   # same stream as print() so output stays in order
    force=True,
)
logger = logging.getLogger("pipeline")

## Step 1 — Create the two decorators

- `@log_action` → logs when a method **starts** and **finishes** (and logs errors if it fails).
- `@log_time` → measures and prints the **execution time**.

`functools.wraps` keeps the original function's name and docstring, so `func.__name__` still says `read_csv` instead of `wrapper`.

In [2]:
def log_action(func):
    """Log when a function/method starts and finishes."""
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        logger.info(f"▶ START  {func.__qualname__}")
        try:
            result = func(*args, **kwargs)
        except Exception as e:
            logger.error(f"✖ FAILED {func.__qualname__}: {e}")
            raise
        logger.info(f"✔ FINISH {func.__qualname__}")
        return result
    return wrapper


def log_time(func):
    """Measure and print how long a function/method takes."""
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"⏱ {func.__qualname__} took {elapsed * 1000:.3f} ms")
        return result
    return wrapper

### Quick test on a plain function

In [3]:
@log_action
@log_time
def slow_add(a, b):
    time.sleep(0.2)
    return a + b

print("Result:", slow_add(2, 3))
print("Name preserved by functools.wraps:", slow_add.__name__)

09:23:11 | INFO | ▶ START  slow_add


⏱ slow_add took 200.203 ms
09:23:12 | INFO | ✔ FINISH slow_add


Result: 5
Name preserved by functools.wraps: slow_add


## Step 2 — Apply the decorators to `CSVReader` and `DataCleaner`

The class logic is exactly the same as before — the only change is the decorator lines above each method. That is the Open/Closed Principle in action.

In [4]:
class CSVReader:
    """Responsible only for loading CSV data."""

    def __init__(self, file_path):
        self.file_path = file_path

    @log_action
    @log_time
    def read_csv(self):
        df = pd.read_csv(self.file_path)
        print(f"Loaded {df.shape[0]} rows × {df.shape[1]} columns")
        return df


class DataCleaner:
    """Responsible only for cleaning data."""

    def __init__(self, df):
        self.df = df.copy()

    @log_action
    @log_time
    def show_missing(self):
        missing = self.df.isnull().sum()
        print("Missing values per column:")
        print(missing[missing > 0])
        return missing

    @log_action
    @log_time
    def fill_missing_with_mean(self):
        num_cols = self.df.select_dtypes(include="number").columns
        self.df[num_cols] = self.df[num_cols].fillna(self.df[num_cols].mean().round(1))
        return self.df

    @log_action
    @log_time
    def remove_duplicates(self):
        before = len(self.df)
        self.df = self.df.drop_duplicates()
        print(f"Removed {before - len(self.df)} duplicate rows")
        return self.df

## Step 3 — Run the pipeline and watch the logs

In [5]:
reader = CSVReader("sample_data.csv")
raw_df = reader.read_csv()
raw_df

09:23:12 | INFO | ▶ START  CSVReader.read_csv


Loaded 7 rows × 7 columns
⏱ CSVReader.read_csv took 2.641 ms
09:23:12 | INFO | ✔ FINISH CSVReader.read_csv


,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,29.0,165.0,68.0,New York,85.0
1,2,Bob,NaN,172.0,NaN,Los Angeles,90.0
2,3,Charlie,35.0,168.0,72.0,Chicago,NaN
3,4,David,NaN,NaN,80.0,Houston,75.0
4,5,Eva,27.0,160.0,55.0,New York,88.0
5,6,Frank,30.0,175.0,85.0,Los Angeles,NaN
6,7,Grace,NaN,162.0,60.0,Chicago,92.0


In [6]:
cleaner = DataCleaner(raw_df)
cleaner.show_missing()
cleaner.fill_missing_with_mean()
clean_df = cleaner.remove_duplicates()
clean_df

09:23:12 | INFO | ▶ START  DataCleaner.show_missing


Missing values per column:
age          3
height_cm    1
weight_kg    1
score        2
dtype: int64
⏱ DataCleaner.show_missing took 1.023 ms
09:23:12 | INFO | ✔ FINISH DataCleaner.show_missing


09:23:12 | INFO | ▶ START  DataCleaner.fill_missing_with_mean


⏱ DataCleaner.fill_missing_with_mean took 2.922 ms
09:23:12 | INFO | ✔ FINISH DataCleaner.fill_missing_with_mean


09:23:12 | INFO | ▶ START  DataCleaner.remove_duplicates


Removed 0 duplicate rows
⏱ DataCleaner.remove_duplicates took 1.050 ms
09:23:12 | INFO | ✔ FINISH DataCleaner.remove_duplicates


,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,29.0,165.0,68.0,New York,85.0
1,2,Bob,30.2,172.0,70.0,Los Angeles,90.0
2,3,Charlie,35.0,168.0,72.0,Chicago,86.0
3,4,David,30.2,167.0,80.0,Houston,75.0
4,5,Eva,27.0,160.0,55.0,New York,88.0
5,6,Frank,30.0,175.0,85.0,Los Angeles,86.0
6,7,Grace,30.2,162.0,60.0,Chicago,92.0


In [7]:
print("Missing values after cleaning:", clean_df.isnull().sum().sum())

Missing values after cleaning: 0


## Step 4 — Open/Closed Principle without touching the class

We can even add timing/logging to a class **we are not allowed to edit** by wrapping its method from the outside:

In [8]:
class Reporter:
    def summary(self, df):
        return df.describe().round(2)

# Extend behaviour from the outside — Reporter's source is unchanged
Reporter.summary = log_action(log_time(Reporter.summary))

Reporter().summary(clean_df)

09:23:12 | INFO | ▶ START  Reporter.summary


⏱ Reporter.summary took 7.039 ms
09:23:12 | INFO | ✔ FINISH Reporter.summary


,id,age,height_cm,weight_kg,score
count,7.00,7.00,7.00,7.00,7.00
mean,4.00,30.23,167.00,70.00,86.00
std,2.16,2.41,5.29,10.47,5.45
min,1.00,27.00,160.00,55.00,75.00
25%,2.50,29.50,163.50,64.00,85.50
50%,4.00,30.20,167.00,70.00,86.00
75%,5.50,30.20,170.00,76.00,89.00
max,7.00,35.00,175.00,85.00,92.00


## Summary

| Decorator | What it adds | Applied to |
|---|---|---|
| `@log_action` | Start / finish (and error) log messages | `read_csv`, `show_missing`, `fill_missing_with_mean`, `remove_duplicates` |
| `@log_time` | Execution time in ms | same methods |

**Why this follows Open/Closed:** logging and timing were added without modifying the bodies of `CSVReader` or `DataCleaner`. The decorators are reusable on any function, and removing them is as easy as deleting one line.

**Stacking order:** with `@log_action` on top of `@log_time`, `log_action` is the outer wrapper — its START prints first and FINISH prints last, and the timer measures only the method itself.

# TP 2: Text Processing, Bag-of-Words and N-gram Language Models

In this TP you build, in Python, the text-processing pipeline that turns raw text into index terms, then use it to build **bag-of-words** vectors, an **inverted index** and a **bigram language model**.

| Exercise | Topic | Points |
|---|---|---|
| 1 | Tokenization with regular expressions | 15 |
| 2 | Normalization, stop words, stemming and lemmatization | 25 |
| 3 | Subword tokenization (Hugging Face) | 10 |
| 4 | Bag-of-words and the inverted index | 20 |
| 5 | Bigram language model, add-k smoothing, perplexity | 30 |

* Complete every cell marked `# TODO`, and write short answers in the **Answer** cells.
* Submit **only this notebook**, renamed `TP2_StudentID_Name_Group.ipynb` for example `TP2_e202000129_DORN_Dawin_I5AMS_A.ipynb`, by email (my email: dorn.dawin@gmail.com) **[25/05/2026 11:59 PM]**.

## 0 · Setup
Run these two cells first. On Google Colab everything is already installed; the first cell only downloads NLTK word lists.

In [ ]:
import nltk
nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

import re, math
from collections import Counter
import pandas as pd

In [ ]:
# Our small collection: 12 short documents about search engines
DOCS = [
    "Search engines index billions of web pages every day.",
    "A search engine ranks the indexed pages for each query.",
    "Users type short queries into the search box.",
    "The crawler downloads web pages and follows their links.",
    "Google's crawler found 3 new pages in the U.S.A. yesterday.",
    "Indexing makes searching fast, but building the index takes time.",
    "The inverted index maps each term to the documents that contain it.",
    "Users don't read every result; they click the top-ranked pages.",
    "Ranking models compare the query with each indexed document.",
    "Stemming maps searching, searched and searches to one term.",
    "Stop words such as 'the' and 'of' are very frequent.",
    "Language models assign a probability to every query.",
]
for i, d in enumerate(DOCS):
    print(i, d)

## Exercise 1 · Tokenization (15 pts)
**Tokenization** splits a text into tokens (words, numbers, punctuation marks). It is the first step of the text transformation component of a search engine (Lecture 1).

### 1.1 · Splitting on spaces (3 pts)
Run the cell. Python's `str.split()` splits on whitespace only.

In [ ]:
print(DOCS[4].split())
print(DOCS[7].split())

**Question 1.1.** Name two problems you see in this output.

**Answer 1.1:** *Two problems are that punctuation stays attached to words, such as pages. and result;*

### 1.2 · A regular-expression tokenizer (6 pts)
The pattern below is adapted from the NLTK example in Jurafsky & Martin, *Speech and Language Processing*, section 2.8. The alternatives are tried **from left to right**.

Complete `tokenize(text)` so that it returns the list of all matches of `PATTERN` in `text` (hint: `re.findall`).

In [ ]:
PATTERN = r"""(?x)
    (?:[A-Z]\.)+            # abbreviations: U.S.A.
  | \w+(?:[-']\w+)*         # words, with optional internal hyphen or apostrophe
  | \$?\d+(?:\.\d+)?%?      # numbers, money, percentages
  | \.\.\.                  # ellipsis
  | [][.,;"'?():_`-]        # single punctuation marks
"""

def tokenize(text):
    # TODO: return the list of tokens
    ...

for i in (4, 7, 10):
    print(tokenize(DOCS[i]))

**Question 1.2.** (a) Compare with 1.1: what is fixed? (b) Why must the abbreviation alternative `(?:[A-Z]\.)+` come **before** the word alternative `\w+…`? (Try swapping them.)

**Answer 1.2:** *(a) The regex separates punctuation from words and keeps abbreviations like U.S.A. together.
(b) The abbreviation pattern must come first so that the tokenizer matches the whole abbreviation before the word pattern can match only part of it.*

### 1.3 · Tokens and types (6 pts)
* A **token** is one occurrence of a word; a **type** is a distinct word (an entry of the vocabulary).
* Tokenize all 12 documents, lower-case every token, and keep only tokens that contain a letter or a digit (drop pure punctuation).
* Print the number of tokens N, the number of types |V|, and the 10 most frequent tokens (`Counter.most_common`).

In [ ]:
# TODO
tokens = ...            # list of lower-cased word tokens of the whole collection
N = ...
V = ...
print("N =", N, " |V| =", V)
print(Counter(tokens).most_common(10))

**Question 1.3.** Which kind of word is the most frequent? Why is that a problem for an index?

**Answer 1.3:** *The words, such as “the,” “and,” and “to,” occur frequently. These words usually provide little information for distinguishing documents.*

## Exercise 2 · Normalization, stop words, stemming (25 pts)
**Normalization** maps tokens that should match to the same term, e.g. *Search*, *search* → *search*. **Stemming** and **lemmatization** reduce word variants (*searching*, *searches*) to a common form.

### 2.1 · Normalization (5 pts)
Complete `normalize(tokens)` with these rules, in this order:
1. lower-case every token;
2. drop tokens that contain no letter and no digit (punctuation);
3. remove a final possessive `'s`: `google's` → `google`;
4. remove the dots of abbreviations: `u.s.a.` → `usa`.

**Check:** `['google', 'crawler', 'found', '3', 'new', 'pages', 'in', 'the', 'usa', 'yesterday']`

In [ ]:
def normalize(tokens):
    out = []

    for t in tokens:
        t = t.lower()

        if not re.search(r"[a-z0-9]", t):
            continue

        t = re.sub(r"'s$", "", t)

        if re.fullmatch(r"(?:[a-z]\.)+[a-z]?\.?", t):
            t = t.replace(".", "")

        out.append(t)

    return out

print(normalize(tokenize(DOCS[4])))

### 2.2 · Stop words (5 pts)
Use NLTK's English stop list. Remove stop words from the normalized tokens of the **whole collection** and report the number of tokens before and after, and the percentage removed.

In [ ]:
from nltk.corpus import stopwords
STOP = set(stopwords.words("english"))
print(len(STOP), "stop words, e.g.", sorted(STOP)[:10])

# TODO
all_terms = [
    term
    for d in DOCS
    for term in normalize(tokenize(d))
]     
kept = [term for term in all_term if term if term not in STOP]            # without stop words
print(len(all_terms), "->", len(kept),
 f"({100 * (1 - len(kept) / len(all_terms)):.1f}% removed)")

**Question 2.2.** Give one search query that would be damaged by removing stop words, and explain why.

**Answer 2.2:** *The query “to be or not to be” would be damaged by stop-word removal because most of its words are common function words.*

### 2.3 · Porter vs Snowball vs lemmatization (8 pts)
Fill a table with one row per word and the columns: `porter`, `snowball`, `lemma_noun` (WordNet lemmatizer, default part of speech = noun) and `lemma_verb` (`pos="v"`).

In [ ]:
from nltk.stem import PorterStemmer, SnowballStemmer, WordNetLemmatizer
porter, snowball, wnl = PorterStemmer(), SnowballStemmer("english"), WordNetLemmatizer()

WORDS = ["searching", "searched", "searches", "indexing", "engines", "queries", "users",
         "libraries", "generously", "university", "universe", "organization", "organ",
         "running", "feet", "was"]

rows = []
for w in WORDS:
    # TODO: append a dict with keys word, porter, snowball, lemma_noun, lemma_verb
    ...
pd.DataFrame(rows)

**Question 2.3.** (a) Give two stems that are not real English words. (b) Give one pair of words with **different meanings** that Porter maps to the same stem, and explain why this hurts precision. (c) Why does `wnl.lemmatize("searching")` return *searching*, but `pos="v"` returns *search*? (d) Give one word where Porter and Snowball disagree.

**Answer 2.3:** *(write your answer here)*

### 2.4 · Your `preprocess` function (7 pts)
Write `preprocess(text, stemmer="porter", remove_stop=True)` that runs the whole pipeline with the functions you wrote above:
`tokenize` → `normalize` → (optional) stop-word removal → stemming (`"porter"`, `"snowball"`, `"lemma"` or `None`), and returns the list of terms.

Later TPs start from this function, so keep it clean: you will copy this cell (possibly updated) into the next notebooks.

**Check:** `preprocess(DOCS[9])` → `['stem', 'map', 'search', 'search', 'search', 'one', 'term']`

In [ ]:
def preprocess(text, stemmer="porter", remove_stop=True):
    # TODO: tokenize -> normalize -> stop words -> stemmer (use porter, snowball, wnl from 2.3)
    ...

for i in (0, 4, 9):
    print(preprocess(DOCS[i]))
print(preprocess(DOCS[9], stemmer="lemma"))
print(preprocess(DOCS[9], stemmer=None, remove_stop=False))

**Question 2.4.** Compare the Porter and lemma outputs for document 9. Which one would let the query *search* match this document best, and why?

**Answer 2.4:** *(write your answer here)*

## Exercise 3 · Subword tokenization (10 pts)
Neural models do not use a fixed list of words. They split text into **subwords** learned from data: frequent words stay whole, rare words are split into frequent pieces, so there are no unknown words. BERT uses the **WordPiece** algorithm; GPT models use **BPE** (byte-pair encoding).

### 3.1 · Load a pretrained tokenizer (4 pts)
Check that you obtain `['i', 'have', 'a', 'new', 'gp', '##u', '!']`.

In [ ]:
from transformers import AutoTokenizer
bert = AutoTokenizer.from_pretrained("bert-base-uncased")
print(bert.tokenize("I have a new GPU!"))

### 3.2 · Explore (6 pts)
Tokenize each word of the list and document 4. For each, print the subword tokens and their number. Then add **three words of your own** (e.g. a rare word, a misspelling, a place name).

In [ ]:
TEST_WORDS = ["search", "searching", "tokenization", "crawler", "bigram", "perplexity", "serach"]
# TODO: add three words of your own, then print each word, its subword tokens and their number
...
# TODO: tokenize DOCS[4] with bert and compare the number of tokens with tokenize(DOCS[4])
...

**Question 3.2.** (a) What does the prefix `##` mean? (b) Which kind of words are split into several pieces, and which stay whole? Why? (c) Why can a subword tokenizer never produce an *unknown word*?

**Answer 3.2:** *(write your answer here)*

## Exercise 4 · Bag-of-words and the inverted index (20 pts)
In the **bag-of-words** model a document is represented by the counts of its terms; word order is ignored.

### 4.1 · Document-term matrix from scratch (8 pts)
1. Preprocess every document with your `preprocess` (Porter, stop words removed).
2. Build the vocabulary: the **sorted** list of all distinct terms.
3. Build the matrix `M` (a list of lists): `M[i][j]` = number of times term `vocab[j]` occurs in document `i`.
4. Print the shape, the percentage of zero cells, and the non-zero entries of document 0.

**Check:** the matrix has 12 rows and 48 columns.

In [ ]:
# TODO
processed = ...
vocab = ...
M = ...

zeros = ...
print("shape:", len(M), "x", len(vocab), f"| zeros: {zeros:.1f}%")
print({vocab[j]: c for j, c in enumerate(M[0]) if c > 0})

### 4.2 · Check with scikit-learn (4 pts)
`CountVectorizer` builds the same matrix. Give it your own pipeline with `analyzer=preprocess`, then check that the vocabulary and the matrix are identical to yours.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
# TODO
cv = ...
X = ...
print("same vocabulary:", ...)
print("same matrix:", ...)

### 4.3 · From the matrix to an inverted index (5 pts)
Build a dictionary `index` that maps each term to its postings list `[(doc_id, tf), ...]`, reading **one column** of `M` per term and keeping only the non-zero cells. Print the postings of `search`, `index`, `page` and `crawler`.

In [ ]:
# TODO
index = ...
for t in ["search", "index", "page", "crawler"]:
    print(t, "->", index[t])

**Question 4.3.** (a) Why does scikit-learn return a *sparse* matrix instead of a normal list of lists? (b) What is the relation between a column of `M` and a postings list?

**Answer 4.3:** *(write your answer here)*

### 4.4 · The limit of bag-of-words (3 pts)
Write **two short sentences with different meanings** that get exactly the same bag-of-words vector, and show it with code.

In [ ]:
# TODO
s1, s2 = ..., ...
print(Counter(preprocess(s1)) == Counter(preprocess(s2)))

**Answer 4.4:** *(write your answer here)*

## Exercise 5 · A bigram language model (30 pts)
A bigram model estimates P(w | previous word) from counts and multiplies these probabilities along a sentence. For language models we **keep stop words and do not stem**: function words matter for word order.

### 5.1 · Counting (6 pts)
* `lm_tokens(text)`: the lower-cased word tokens of `text` (use `tokenize`, drop pure punctuation).
* Training sentences: each document becomes `["<s>"] + lm_tokens(doc) + ["</s>"]`.
* Count bigrams in `bigram_c`, and in `context_c` count each word **as the first word of a bigram** (every token except the last `</s>`).
* `V` = number of distinct word types in the training sentences, **including `</s>` but not `<s>`**.

**Check:** V = 78, 122 bigram types, C(&lt;s&gt;) = 12.

In [ ]:
def lm_tokens(text):
    # TODO
    ...

train = [["<s>"] + lm_tokens(d) + ["</s>"] for d in DOCS]
bigram_c, context_c = Counter(), Counter()
for s in train:
    # TODO: update both counters
    ...
V = ...
print("V =", V, "| bigram types:", len(bigram_c), "| C(<s>) =", context_c["<s>"])
print(train[3])

### 5.2 · MLE probabilities (6 pts)
P_MLE(w | prev) = C(prev w) / C(prev). Write `p_mle(w, prev)` and `sentence_prob(text, p)`, which pads the sentence with `<s>` and `</s>` and multiplies the bigram probabilities given by the function `p`. Compute the probability of the three sentences.

In [ ]:
def p_mle(w, prev):
    # TODO
    ...

def sentence_prob(text, p):
    # TODO
    ...

for s in ["the crawler downloads web pages",
          "the crawler downloads web pages every day",
          "pages web downloads crawler the"]:
    print(f"{sentence_prob(s, p_mle):.6f}  {s}")

**Question 5.2.** The second sentence is not in the collection. Why is its probability not zero? Why is the third one zero although it uses the same words as the first?

**Answer 5.2:** *(write your answer here)*

### 5.3 · Add-k smoothing (6 pts)
P_add-k(w | prev) = (C(prev w) + k) / (C(prev) + k·V). Write `p_addk(w, prev, k)` and check that P(engine | search) = 2/81 ≈ 0.0247 for k = 1.

In [ ]:
def p_addk(w, prev, k):
    # TODO
    ...

print(p_mle("engine", "search"), p_addk("engine", "search", 1), p_addk("engine", "search", 0.1))
print(sentence_prob("pages web downloads crawler the", lambda w, prev: p_addk(w, prev, 1)))

**Question 5.3.** P(engine | search) is 1/3 with MLE but only 0.025 with k = 1. Where did the probability go?

**Answer 5.3:** *(write your answer here)*

### 5.4 · Perplexity on held-out sentences (8 pts)
PP = exp(−(1/N) Σ log P(w | prev)), where N counts every predicted token, including `</s>`. Use **log probabilities** (sum of logs) to avoid underflow; return `float("inf")` if a probability is 0.

Compute the perplexity of the held-out sentences with MLE and with add-k for k ∈ {1, 0.5, 0.1, 0.05, 0.01, 0.001}, and show the results in a table.

In [ ]:
HELD_OUT = ["the search engine ranks pages for users",
            "users click the top pages",
            "the crawler follows links to new pages"]

def perplexity(sentences, p):
    # TODO
    ...

print("MLE:", perplexity(HELD_OUT, p_mle))
# TODO: table of k and perplexity

**Question 5.4.** (a) Why is the MLE perplexity infinite? (b) Which k gives the lowest perplexity? (c) Explain why a **large** k and a **very small** k are both worse.

**Answer 5.4:** *(write your answer here)*

## Exercise 5 · A complete CSV cleaning pipeline

Build a reusable workflow with four design patterns:

| Pattern | Responsibility |
| :--- | :--- |
| **Template Method** | `DataPipeline.run()` fixes the order: load → clean → transform → save. |
| **Strategy** | `DataCleaner` delegates missing-value handling to an interchangeable strategy. |
| **Factory** | `TransformFactory` creates transformations from configuration names. |
| **Decorator** | `@log_action` and `@log_time` provide optional execution diagnostics. |

Run all cells in order. The example writes `dirty_sample.csv` and `cleaned_sample.csv`
in the notebook's working directory. Verbose diagnostics are disabled by default.

### 1. Imports and execution decorators

Set `ENABLE_LOGGING = True` to inspect calls, failures, and elapsed time.
Both decorators preserve the wrapped function's metadata; timing is reported even
when a call fails.

In [1]:
from abc import ABC, abstractmethod
from functools import wraps
from pathlib import Path
from time import perf_counter

import pandas as pd
from IPython.display import display

ENABLE_LOGGING = False


def log_action(func):
    """Optionally report the start, success, or failure of a call."""
    @wraps(func)
    def wrapper(*args, **kwargs):
        if ENABLE_LOGGING:
            print(f"[LOG] {func.__qualname__} started")
        try:
            result = func(*args, **kwargs)
        except Exception as exc:
            if ENABLE_LOGGING:
                print(f"[LOG] {func.__qualname__} failed: {exc}")
            raise
        if ENABLE_LOGGING:
            print(f"[LOG] {func.__qualname__} finished")
        return result
    return wrapper


def log_time(func):
    """Optionally report elapsed time, including unsuccessful calls."""
    @wraps(func)
    def wrapper(*args, **kwargs):
        if not ENABLE_LOGGING:
            return func(*args, **kwargs)
        start = perf_counter()
        try:
            return func(*args, **kwargs)
        finally:
            print(f"[TIME] {func.__qualname__}: {perf_counter() - start:.4f}s")
    return wrapper

### 2. CSV reader

Encapsulate CSV ingestion and provide a notebook-friendly preview.

In [2]:
class CSVReader:
    def __init__(self, file_path: str | Path):
        self.file_path = Path(file_path)
        self.df: pd.DataFrame | None = None

    @log_action
    @log_time
    def read(self) -> pd.DataFrame:
        self.df = pd.read_csv(self.file_path)
        return self.df

    def preview(self, n: int = 5) -> None:
        if self.df is None:
            raise RuntimeError("Call read() before preview().")
        display(self.df.head(n))

### 3. Missing-value strategies · Strategy Pattern

- **`DropMissing`** removes rows containing any missing cell.
- **`FillMean`** fills numeric columns with their means and preserves missing text.
- **`FillMode`** fills each column with its first mode; ties follow pandas' mode ordering.

All strategies return a new DataFrame. A column containing only missing values has
no mean or mode, so it remains missing. Empty or whitespace-only strings are not
treated as missing by these strategies.

In [3]:
class MissingValueStrategy(ABC):
    @abstractmethod
    def handle(self, df: pd.DataFrame) -> pd.DataFrame:
        """Return cleaned data without changing the input DataFrame."""
        ...


class DropMissing(MissingValueStrategy):
    def handle(self, df: pd.DataFrame) -> pd.DataFrame:
        return df.dropna().reset_index(drop=True)


class FillMean(MissingValueStrategy):
    def handle(self, df: pd.DataFrame) -> pd.DataFrame:
        return df.fillna(df.mean(numeric_only=True))


class FillMode(MissingValueStrategy):
    def handle(self, df: pd.DataFrame) -> pd.DataFrame:
        # Only calculate modes for columns that need imputation.
        replacements = {}
        for name in df.columns[df.isna().any()]:
            modes = df[name].mode()
            if not modes.empty:
                replacements[name] = modes.iloc[0]
        return df.fillna(replacements)

### 4. Cleaning context

`DataCleaner` delegates to the selected strategy. With no strategy, it leaves the
data unchanged and avoids allocating an unnecessary copy.

In [4]:
class DataCleaner:
    def __init__(self, strategy: MissingValueStrategy | None = None):
        self.strategy = strategy

    @log_action
    @log_time
    def clean(self, df: pd.DataFrame) -> pd.DataFrame:
        return df if self.strategy is None else self.strategy.handle(df)

### 5. Reusable transformations

- **`NormalizeColumns`** strips headers, lowercases them, and collapses whitespace
  to underscores. It rejects headers that would become ambiguous duplicates.
- **`StandardizeText`** strips and lowercases text while preserving missing values
  and non-string values in mixed object columns.
- **`RemoveDuplicates`** removes identical rows and resets the index.

Order matters: standardize text before removing duplicates so cosmetic differences
do not hide duplicate records. Each transformation preserves its input.

In [5]:
class DataTransform(ABC):
    @abstractmethod
    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        """Return transformed data without changing the input DataFrame."""
        ...


class NormalizeColumns(DataTransform):
    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        columns = (pd.Index(df.columns).astype(str).str.strip().str.lower()
                   .str.replace(r"\s+", "_", regex=True))
        if columns.has_duplicates:
            duplicates = columns[columns.duplicated()].unique().tolist()
            raise ValueError(f"Column normalization creates duplicate names: {duplicates}")
        return df.set_axis(columns, axis="columns")


class RemoveDuplicates(DataTransform):
    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        return df.drop_duplicates().reset_index(drop=True)


class StandardizeText(DataTransform):
    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        result = df.copy()
        for name in result.select_dtypes(include=["object", "string"]).columns:
            column = result[name]
            if pd.api.types.infer_dtype(column, skipna=True) in {"string", "empty"}:
                result[name] = column.str.strip().str.lower()
            else:
                # The .str accessor alone would replace non-string values with NaN.
                result[name] = column.map(
                    lambda value: value.strip().lower() if isinstance(value, str) else value
                )
        return result

### 6. Transformation factory · Factory Pattern

Create the requested transformation from its name. Unknown names raise a clear error.

In [6]:
class TransformFactory:
    _transforms = {
        "normalize_columns": NormalizeColumns,
        "standardize_text": StandardizeText,
        "remove_duplicates": RemoveDuplicates,
    }

    @classmethod
    def get_transform(cls, name: str) -> DataTransform:
        key = name.strip().lower()
        if key not in cls._transforms:
            available = ", ".join(cls._transforms)
            raise ValueError(f"Unknown transform {name!r}. Available: {available}")
        return cls._transforms[key]()

### 7. Pipeline template · Template Method Pattern

The base class defines the workflow once. Concrete subclasses implement the four
steps. `run()` returns the pipeline so callers can inspect its result and summary.

In [7]:
class DataPipeline(ABC):
    @log_action
    @log_time
    def run(self):
        """Execute the fixed load → clean → transform → save sequence."""
        self.load()
        self.clean()
        self.transform()
        self.save()
        return self

    @abstractmethod
    def load(self) -> None:
        ...

    @abstractmethod
    def clean(self) -> None:
        ...

    @abstractmethod
    def transform(self) -> None:
        ...

    @abstractmethod
    def save(self) -> None:
        ...

### 8. Concrete CSV pipeline

Implement the four template steps using the reader, cleaner, and transformation factory.

In [8]:
class CSVDataPipeline(DataPipeline):
    def __init__(self, input_path, output_path, missing_strategy=None, transforms=None):
        self.input_path = input_path
        self.output_path = output_path
        self.cleaner = DataCleaner(missing_strategy)
        self.transforms = tuple(transforms) if transforms is not None else ()
        self.df = None

    def load(self):
        self.df = CSVReader(self.input_path).read()

    def clean(self):
        self.df = self.cleaner.clean(self.df)

    def transform(self):
        for name in self.transforms:
            transformer = TransformFactory.get_transform(name)
            self.df = transformer.apply(self.df)

    def save(self):
        self.df.to_csv(self.output_path, index=False)

### 9. Create the sample dataset

The five input rows contain inconsistent headers and text casing, one duplicate
Alice record, and three missing cells: age, salary, and department.

In [9]:
INPUT_PATH = Path("dirty_sample.csv")
OUTPUT_PATH = Path("cleaned_sample.csv")

raw_df = pd.DataFrame({
    " Full Name ": [" Alice ", "BOB", "Charlie ", " Alice ", "David"],
    " Age ": [25.0, None, 30.0, 25.0, 40.0],
    " Salary ": [50000.0, 60000.0, None, 50000.0, 75000.0],
    " Department ": [" HR ", "engineering", "FINANCE", " HR ", None],
})
raw_df.to_csv(INPUT_PATH, index=False)

### 10. Run the pipeline

Run the four steps and compare the original and cleaned data using standard pandas tables.

`FillMean` fills age with **30** and salary with **58,750** before deduplication.
David's department remains missing because it is not numeric.

In [10]:
pipeline = CSVDataPipeline(
    input_path=INPUT_PATH,
    output_path=OUTPUT_PATH,
    missing_strategy=FillMean(),
    transforms=["normalize_columns", "standardize_text", "remove_duplicates"],
).run()

cleaned_df = pipeline.df
print("Before cleaning:")
display(raw_df)
print("After cleaning:")
display(cleaned_df)

Before cleaning:


,Full Name,Age,Salary,Department
0,Alice,25.0,50000.0,HR
1,BOB,NaN,60000.0,engineering
2,Charlie,30.0,NaN,FINANCE
3,Alice,25.0,50000.0,HR
4,David,40.0,75000.0,NaN


After cleaning:


,full_name,age,salary,department
0,alice,25.0,50000.0,hr
1,bob,30.0,60000.0,engineering
2,charlie,30.0,58750.0,finance
3,david,40.0,75000.0,NaN
